# TaskManager behavior with LaplaceSL BEM operator

This notebook tests whether `TaskManager` can be used with `LaplaceSL` from `ngsolve.bem`.

**Observation**: Without `TaskManager`, results are bit-for-bit identical across runs. With `TaskManager`, results vary slightly (~1.4%).

**Environment**: NGSolve 6.2.2602 (pip), Python 3.12.8, Windows Server 2022 (8-core Xeon)

## 1. Setup: Create torus mesh with OCC

In [1]:
from ngsolve import Mesh, HDivSurface, TaskManager, ds
from ngsolve.bem import LaplaceSL
from netgen.occ import WorkPlane, Axes, Axis, Pnt, Dir, OCCGeometry
from netgen.meshing import MeshingParameters
import numpy as np

MU_0 = 4e-7 * np.pi
R, a = 0.1, 0.01  # torus: major radius 100mm, minor radius 10mm

# Create torus mesh (OCC native, no Cubit dependency)
wp = WorkPlane(Axes(p=Pnt(R, 0, 0), n=Dir(0, 1, 0), h=Dir(0, 0, 1)))
circle = wp.Circle(a).Face()
torus = circle.Revolve(Axis(p=Pnt(0, 0, 0), d=Dir(0, 0, 1)), 360)
geo = OCCGeometry(torus)
ngmesh = geo.GenerateMesh(
    mp=MeshingParameters(maxh=1.0, curvaturesafety=0.5, segmentsperedge=2)
)
mesh = Mesh(ngmesh)

fes = HDivSurface(mesh, order=0)
u, v = fes.TnT()
ndof = fes.ndof
print(f"HDivSurface DOFs: {ndof}")

# Analytical reference (Neumann formula, external inductance)
L_neumann = MU_0 * R * (np.log(8 * R / a) - 2)
print(f"Neumann formula: {L_neumann*1e9:.1f} nH")

HDivSurface DOFs: 269
Neumann formula: 299.3 nH


## 2. Helper: extract self-inductance from LaplaceSL

Build the BEM operator, extract the dense matrix column-by-column, and compute self-inductance via $L = 1 / (\mathbf{e}^T \mathbf{L}^{-1} \mathbf{e})$ under a uniform current distribution.

In [2]:
import time

def extract_inductance(use_taskmanager):
    """Build LaplaceSL and extract self-inductance. Returns (L_total, t_build, t_extract)."""
    t0 = time.perf_counter()
    if use_taskmanager:
        with TaskManager():
            L_op = LaplaceSL(u.Trace() * ds) * v.Trace() * ds
    else:
        L_op = LaplaceSL(u.Trace() * ds) * v.Trace() * ds
    t_build = time.perf_counter() - t0

    # Dense matrix extraction (column-by-column)
    t0 = time.perf_counter()
    L = np.zeros((ndof, ndof))
    ei = L_op.mat.CreateColVector()
    col = L_op.mat.CreateColVector()
    for j in range(ndof):
        ei[:] = 0; ei[j] = 1.0
        L_op.mat.Mult(ei, col)
        L[:, j] = col.FV().NumPy()
    L *= MU_0
    t_extract = time.perf_counter() - t0

    # Self-inductance from uniform current
    e = np.ones(ndof) / ndof
    L_total = 1.0 / (e @ np.linalg.solve(L, e))
    return L_total, t_build, t_extract

## 3. Without TaskManager (deterministic)

In [3]:
print("Without TaskManager:")
results_no_tm = []
times_no_tm = []
for i in range(5):
    L, t_build, t_extract = extract_inductance(use_taskmanager=False)
    results_no_tm.append(L)
    times_no_tm.append((t_build, t_extract))
    print(f"  Run {i}: L = {L*1e9:.1f} nH  (build {t_build:.3f}s, extract {t_extract:.3f}s)")

print(f"\n  Spread: {(max(results_no_tm)-min(results_no_tm))*1e9:.2f} nH")

Without TaskManager:


  Run 0: L = 369.1 nH  (build 0.013s, extract 1.417s)


  Run 1: L = 369.1 nH  (build 0.014s, extract 1.373s)


  Run 2: L = 369.1 nH  (build 0.014s, extract 1.360s)


  Run 3: L = 369.1 nH  (build 0.014s, extract 1.371s)


  Run 4: L = 369.1 nH  (build 0.013s, extract 1.375s)

  Spread: 0.00 nH


## 4. With TaskManager (results vary between runs)

In [4]:
print("With TaskManager:")
results_tm = []
times_tm = []
for i in range(5):
    L, t_build, t_extract = extract_inductance(use_taskmanager=True)
    results_tm.append(L)
    times_tm.append((t_build, t_extract))
    print(f"  Run {i}: L = {L*1e9:.1f} nH  (build {t_build:.3f}s, extract {t_extract:.3f}s)")

spread = (max(results_tm) - min(results_tm)) * 1e9
std = np.std(results_tm) * 1e9
print(f"\n  Spread: {spread:.2f} nH, Std: {std:.2f} nH")

With TaskManager:


  Run 0: L = 369.7 nH  (build 0.007s, extract 1.359s)


  Run 1: L = 371.9 nH  (build 0.004s, extract 1.379s)


  Run 2: L = 369.4 nH  (build 0.005s, extract 1.369s)


  Run 3: L = 372.5 nH  (build 0.006s, extract 1.364s)


  Run 4: L = 372.7 nH  (build 0.004s, extract 1.364s)

  Spread: 3.35 nH, Std: 1.40 nH


## 5. Timing comparison

The bottleneck is `mat.Mult()` (dense extraction), not operator construction. Each mat-vec may be triggering on-the-fly BEM integration rather than reading from a preassembled dense matrix; this should be confirmed against the recommended ngsbem workflow/API. TaskManager does not speed up either phase.

In [5]:
avg_build_no = np.mean([t[0] for t in times_no_tm])
avg_extract_no = np.mean([t[1] for t in times_no_tm])
avg_build_tm = np.mean([t[0] for t in times_tm])
avg_extract_tm = np.mean([t[1] for t in times_tm])

print(f"{'':20s} {'Build':>10s} {'Extract':>10s} {'Total':>10s}")
print(f"{'-'*52}")
print(f"{'No TaskManager':20s} {avg_build_no:10.3f}s {avg_extract_no:10.3f}s {avg_build_no+avg_extract_no:10.3f}s")
print(f"{'With TaskManager':20s} {avg_build_tm:10.3f}s {avg_extract_tm:10.3f}s {avg_build_tm+avg_extract_tm:10.3f}s")
print(f"\nmat.Mult() per call: {avg_extract_no/ndof*1000:.1f} ms  ({ndof} calls)")

                          Build    Extract      Total
----------------------------------------------------
No TaskManager            0.014s      1.379s      1.393s
With TaskManager          0.005s      1.367s      1.372s

mat.Mult() per call: 5.1 ms  (269 calls)


## 6. Root cause isolation: thread count vs fluctuation

Setting `SetNumThreads(1)` inside `TaskManager` eliminates the fluctuation entirely, indicating that the variation comes from non-deterministic floating-point summation order in multi-threaded BEM integration.

In [6]:
from ngsolve import SetNumThreads

def run_n_times(use_tm, nruns=5):
    results = []
    for _ in range(nruns):
        L, _, _ = extract_inductance(use_tm)
        results.append(L * 1e9)
    arr = np.array(results)
    return arr

print(f"{'Config':30s} {'Mean':>10s} {'Spread':>10s}")
print("-" * 52)

# No TaskManager (reference)
r = run_n_times(False)
print(f"{'No TaskManager':30s} {r.mean():10.2f} {r.max()-r.min():10.3f} nH")

# TaskManager with varying thread counts
for nt in [1, 2, 4, 8]:
    SetNumThreads(nt)
    r = run_n_times(True)
    print(f"{'TaskManager (nthreads=' + str(nt) + ')':30s} {r.mean():10.2f} {r.max()-r.min():10.3f} nH")

Config                               Mean     Spread
----------------------------------------------------


No TaskManager                     369.14      0.000 nH


TaskManager (nthreads=1)           369.14      0.000 nH


TaskManager (nthreads=2)           369.35      1.349 nH


TaskManager (nthreads=4)           368.83      1.265 nH


TaskManager (nthreads=8)           370.55      5.411 nH


## 7. Matrix diagnostics

Check the condition number and symmetry of the extracted BEM matrix. The single-layer potential matrix should be symmetric by construction (Galerkin BEM).

In [7]:
# Extract reference matrix (no TaskManager)
L_op_ref = LaplaceSL(u.Trace() * ds) * v.Trace() * ds
L_ref = np.zeros((ndof, ndof))
ei = L_op_ref.mat.CreateColVector()
col = L_op_ref.mat.CreateColVector()
for j in range(ndof):
    ei[:] = 0; ei[j] = 1.0; L_op_ref.mat.Mult(ei, col)
    L_ref[:, j] = col.FV().NumPy()
L_ref *= MU_0

cond = np.linalg.cond(L_ref)
sym_err = np.linalg.norm(L_ref - L_ref.T) / np.linalg.norm(L_ref)
neg_diag = int(np.sum(np.diag(L_ref) < 0))

print(f"Matrix size:     {ndof} x {ndof}")
print(f"Condition number: {cond:.1f}")
print(f"Symmetry error:  ||L - L^T|| / ||L|| = {sym_err:.2e}")
print(f"Negative diag:   {neg_diag}")

# Compare TaskManager matrix against reference
SetNumThreads(8)
with TaskManager():
    L_op_tm = LaplaceSL(u.Trace() * ds) * v.Trace() * ds
L_tm = np.zeros((ndof, ndof))
ei = L_op_tm.mat.CreateColVector()
col = L_op_tm.mat.CreateColVector()
for j in range(ndof):
    ei[:] = 0; ei[j] = 1.0; L_op_tm.mat.Mult(ei, col)
    L_tm[:, j] = col.FV().NumPy()
L_tm *= MU_0

diff = np.linalg.norm(L_tm - L_ref) / np.linalg.norm(L_ref)
print(f"\n||L_tm - L_ref|| / ||L_ref|| = {diff:.2e}  (TaskManager vs reference)")

Matrix size:     269 x 269
Condition number: 518.3
Symmetry error:  ||L - L^T|| / ||L|| = 6.40e-02
Negative diag:   0



||L_tm - L_ref|| / ||L_ref|| = 5.51e-03  (TaskManager vs reference)
